# Análise Exploratória de Dados (EDA) — Datathon Fase 05## Previsão de Preços de Ações: PETR4.SA, VALE3.SA, ITUB4.SA**Objetivo**: Entender o comportamento histórico das ações, identificar padrões,sazonalidades e correlações que alimentarão o modelo LSTM e o agente inteligente.**Dados**: Coletados via yfinance (2018-01 até 2026-05) | ~2081 registros por ação.

In [ ]:
# Imports e configuraçãoimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Path# Configurações de visualizaçãoplt.style.use('seaborn-v0_8-whitegrid')sns.set_palette("husl")plt.rcParams['figure.figsize'] = (14, 6)plt.rcParams['font.size'] = 11# Carregar dadosDATA_DIR = Path("../data/raw")petr4 = pd.read_csv(DATA_DIR / "PETR4_SA_historico.csv", index_col=0, parse_dates=True)vale3 = pd.read_csv(DATA_DIR / "VALE3_SA_historico.csv", index_col=0, parse_dates=True)itub4 = pd.read_csv(DATA_DIR / "ITUB4_SA_historico.csv", index_col=0, parse_dates=True)# Dataset combinadocombined = pd.read_csv(DATA_DIR / "combined_close_prices.csv", index_col=0, parse_dates=True)print(f"PETR4: {petr4.shape} | {petr4.index[0].strftime('%Y-%m-%d')} → {petr4.index[-1].strftime('%Y-%m-%d')}")print(f"VALE3: {vale3.shape} | {vale3.index[0].strftime('%Y-%m-%d')} → {vale3.index[-1].strftime('%Y-%m-%d')}")print(f"ITUB4: {itub4.shape} | {itub4.index[0].strftime('%Y-%m-%d')} → {itub4.index[-1].strftime('%Y-%m-%d')}")

## 1. Visão Geral dos Dados

In [ ]:
# Estatísticas descritivas - PETR4print("=" * 60)print("PETR4.SA — Estatísticas Descritivas")print("=" * 60)print(petr4.describe().round(2))print(f"\nMissing values:\n{petr4.isnull().sum()}")print(f"\nDias com volume zero: {(petr4['Volume'] == 0).sum()}")

## 2. Série Temporal Completa com Tendências

In [ ]:
# Série temporal de preços de fechamento com médias móveisfig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)for ax, (name, df) in zip(axes, [("PETR4.SA", petr4), ("VALE3.SA", vale3), ("ITUB4.SA", itub4)]):    ax.plot(df.index, df['Close'], alpha=0.7, linewidth=0.8, label='Close')    ax.plot(df.index, df['Close'].rolling(30).mean(), linewidth=1.5, label='SMA 30')    ax.plot(df.index, df['Close'].rolling(90).mean(), linewidth=1.5, label='SMA 90')    ax.set_title(f'{name} — Preço de Fechamento + Médias Móveis', fontsize=12)    ax.set_ylabel('Preço (R$)')    ax.legend(loc='upper left')    ax.grid(True, alpha=0.3)axes[-1].set_xlabel('Data')plt.tight_layout()plt.savefig('../docs/img/serie_temporal_completa.png', dpi=150, bbox_inches='tight')plt.show()

## 3. Retornos Diários e Volatilidade

In [ ]:
# Calcular retornos diários logarítmicospetr4['returns'] = np.log(petr4['Close'] / petr4['Close'].shift(1))vale3['returns'] = np.log(vale3['Close'] / vale3['Close'].shift(1))itub4['returns'] = np.log(itub4['Close'] / itub4['Close'].shift(1))fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)for ax, (name, df) in zip(axes, [("PETR4.SA", petr4), ("VALE3.SA", vale3), ("ITUB4.SA", itub4)]):    ax.plot(df.index, df['returns'], alpha=0.6, linewidth=0.5)    ax.axhline(y=0, color='black', linewidth=0.5)    ax.set_title(f'{name} — Retornos Diários (log)', fontsize=12)    ax.set_ylabel('Retorno')axes[-1].set_xlabel('Data')plt.tight_layout()plt.savefig('../docs/img/retornos_diarios.png', dpi=150, bbox_inches='tight')plt.show()# Estatísticas dos retornosprint("\nEstatísticas dos Retornos Diários:")print("-" * 50)for name, df in [("PETR4", petr4), ("VALE3", vale3), ("ITUB4", itub4)]:    r = df['returns'].dropna()    print(f"{name}: média={r.mean():.4f} | std={r.std():.4f} | "          f"skew={r.skew():.2f} | kurt={r.kurtosis():.2f} | "          f"vol_anual={r.std() * np.sqrt(252):.2%}")

## 4. Volatilidade Rolling (30 dias)

In [ ]:
# Volatilidade rollingfig, ax = plt.subplots(figsize=(14, 6))for name, df in [("PETR4", petr4), ("VALE3", vale3), ("ITUB4", itub4)]:    vol = df['returns'].rolling(30).std() * np.sqrt(252)    ax.plot(df.index, vol, label=f'{name} (vol 30d anualizada)', linewidth=1)ax.set_title('Volatilidade Rolling 30 dias (Anualizada)', fontsize=13)ax.set_ylabel('Volatilidade')ax.set_xlabel('Data')ax.legend()ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('../docs/img/volatilidade_rolling.png', dpi=150, bbox_inches='tight')plt.show()

## 5. Drawdown (Queda Máxima do Pico)

In [ ]:
# Drawdownfig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)for ax, (name, df) in zip(axes, [("PETR4", petr4), ("VALE3", vale3), ("ITUB4", itub4)]):    cummax = df['Close'].cummax()    drawdown = (df['Close'] - cummax) / cummax    ax.fill_between(df.index, drawdown, 0, alpha=0.4, color='red')    ax.set_title(f'{name} — Drawdown', fontsize=12)    ax.set_ylabel('Drawdown (%)')    max_dd = drawdown.min()    max_dd_date = drawdown.idxmin()    ax.annotate(f'Max: {max_dd:.1%}', xy=(max_dd_date, max_dd),                fontsize=9, color='darkred', fontweight='bold')axes[-1].set_xlabel('Data')plt.tight_layout()plt.savefig('../docs/img/drawdown.png', dpi=150, bbox_inches='tight')plt.show()# Resumo drawdownsprint("\nDrawdown Máximo:")print("-" * 40)for name, df in [("PETR4", petr4), ("VALE3", vale3), ("ITUB4", itub4)]:    cummax = df['Close'].cummax()    drawdown = (df['Close'] - cummax) / cummax    print(f"  {name}: {drawdown.min():.1%} em {drawdown.idxmin().strftime('%Y-%m-%d')}")

## 6. Sazonalidade — Mês e Dia da Semana

In [ ]:
# Sazonalidade mensalfig, axes = plt.subplots(1, 2, figsize=(14, 5))# Retorno médio por mêspetr4['month'] = petr4.index.monthmonthly_returns = petr4.groupby('month')['returns'].mean()axes[0].bar(monthly_returns.index, monthly_returns.values, color='steelblue', alpha=0.7)axes[0].axhline(y=0, color='black', linewidth=0.5)axes[0].set_title('PETR4 — Retorno Médio por Mês', fontsize=12)axes[0].set_xlabel('Mês')axes[0].set_ylabel('Retorno Médio')axes[0].set_xticks(range(1, 13))# Retorno médio por dia da semanapetr4['weekday'] = petr4.index.dayofweekweekday_returns = petr4.groupby('weekday')['returns'].mean()days = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex']axes[1].bar(range(5), weekday_returns.values[:5], color='coral', alpha=0.7)axes[1].axhline(y=0, color='black', linewidth=0.5)axes[1].set_title('PETR4 — Retorno Médio por Dia da Semana', fontsize=12)axes[1].set_xlabel('Dia')axes[1].set_ylabel('Retorno Médio')axes[1].set_xticks(range(5))axes[1].set_xticklabels(days)plt.tight_layout()plt.savefig('../docs/img/sazonalidade.png', dpi=150, bbox_inches='tight')plt.show()# Volatilidade por mêsprint("\nVolatilidade PETR4 por Mês (desvio padrão retornos):")monthly_vol = petr4.groupby('month')['returns'].std()for m, v in monthly_vol.items():    bar = '█' * int(v * 500)    print(f"  Mês {m:02d}: {v:.4f} {bar}")

## 7. Correlação entre Ações

In [ ]:
# Correlação entre retornosreturns_df = pd.DataFrame({    'PETR4': petr4['returns'],    'VALE3': vale3['returns'],    'ITUB4': itub4['returns'],}).dropna()fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Heatmap de correlaçãocorr = returns_df.corr()sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0, ax=axes[0],            vmin=-1, vmax=1, fmt='.3f', square=True)axes[0].set_title('Correlação dos Retornos Diários', fontsize=12)# Correlação rolling 60 diasrolling_corr = returns_df['PETR4'].rolling(60).corr(returns_df['VALE3'])axes[1].plot(returns_df.index, rolling_corr, linewidth=1, label='PETR4 vs VALE3')rolling_corr2 = returns_df['PETR4'].rolling(60).corr(returns_df['ITUB4'])axes[1].plot(returns_df.index, rolling_corr2, linewidth=1, label='PETR4 vs ITUB4')axes[1].axhline(y=0, color='black', linewidth=0.5, linestyle='--')axes[1].set_title('Correlação Rolling 60 dias', fontsize=12)axes[1].set_xlabel('Data')axes[1].set_ylabel('Correlação')axes[1].legend()plt.tight_layout()plt.savefig('../docs/img/correlacao.png', dpi=150, bbox_inches='tight')plt.show()print(f"\nCorrelação estática (período completo):")print(corr.to_string())

## 8. Distribuição dos Retornos

In [ ]:
# Distribuição dos retornosfig, axes = plt.subplots(1, 3, figsize=(14, 4))for ax, (name, df) in zip(axes, [("PETR4", petr4), ("VALE3", vale3), ("ITUB4", itub4)]):    r = df['returns'].dropna()    ax.hist(r, bins=80, density=True, alpha=0.7, color='steelblue', edgecolor='white')    ax.axvline(x=r.mean(), color='red', linestyle='--', label=f'μ={r.mean():.4f}')    ax.axvline(x=r.mean() + 2*r.std(), color='orange', linestyle=':', alpha=0.7)    ax.axvline(x=r.mean() - 2*r.std(), color='orange', linestyle=':', alpha=0.7)    ax.set_title(f'{name} — Distribuição Retornos', fontsize=11)    ax.set_xlabel('Retorno')    ax.legend(fontsize=9)plt.tight_layout()plt.savefig('../docs/img/distribuicao_retornos.png', dpi=150, bbox_inches='tight')plt.show()

## 9. Insights de NegócioCom base na análise exploratória, identificamos os seguintes insights:### Insight 1: Valorização expressiva com alta volatilidadePETR4 saiu de ~R$3 (2018) para ~R$46 (2026), uma valorização de ~1400%.Porém, passou por drawdowns superiores a 50%, indicando risco significativo.O modelo precisa capturar tanto tendência quanto volatilidade.### Insight 2: Volatilidade em clusters (agrupada no tempo)Períodos de alta volatilidade são persistentes (COVID-2020, eleições 2022).Isso valida o uso de LSTM que captura dependências temporais de longo prazo.O agente deve alertar o usuário quando volatilidade está acima da média histórica.### Insight 3: Correlação moderada entre setoresPETR4 (petróleo) e VALE3 (mineração) mostram correlação moderada (~0.5-0.7),ambas commodities. ITUB4 (bancário) tem correlação menor.Diversificação entre setores reduz risco — info útil para o agente.### Insight 4: Sazonalidade mensal observávelMeses com divulgação de resultados (março, maio, agosto, novembro) tendema ter maior volatilidade. O modelo pode se beneficiar de features de calendário.### Insight 5: Caudas pesadas nos retornos (fat tails)A distribuição dos retornos tem curtose elevada (leptocúrtica), indicandoque eventos extremos são mais frequentes que uma distribuição normal preveria.Risk management precisa considerar VaR com distribuições não-normais.

In [ ]:
# Resumo quantitativo dos insightsprint("=" * 60)print("RESUMO QUANTITATIVO DOS INSIGHTS")print("=" * 60)# Insight 1: Valorizaçãovaloriz = (petr4['Close'].iloc[-1] / petr4['Close'].iloc[0] - 1) * 100print(f"\n1. Valorização PETR4: {valoriz:.0f}%")# Insight 2: Clusters de volatilidadevol_30d = petr4['returns'].rolling(30).std() * np.sqrt(252)vol_media = vol_30d.mean()periodos_alta_vol = (vol_30d > vol_media * 1.5).sum()print(f"2. Dias com vol > 1.5x média: {periodos_alta_vol} ({periodos_alta_vol/len(petr4)*100:.1f}%)")# Insight 3: Correlaçãoprint(f"3. Correlação PETR4-VALE3: {returns_df['PETR4'].corr(returns_df['VALE3']):.3f}")print(f"   Correlação PETR4-ITUB4: {returns_df['PETR4'].corr(returns_df['ITUB4']):.3f}")# Insight 4: Volatilidade por mêsvol_by_month = petr4.groupby('month')['returns'].std()print(f"4. Mês mais volátil: {vol_by_month.idxmax()} (std={vol_by_month.max():.4f})")print(f"   Mês menos volátil: {vol_by_month.idxmin()} (std={vol_by_month.min():.4f})")# Insight 5: Curtosekurt = petr4['returns'].dropna().kurtosis()print(f"5. Curtose dos retornos: {kurt:.2f} (normal=0, >0 = caudas pesadas)")print(f"   Skewness: {petr4['returns'].dropna().skew():.2f}")

## 10. Conclusão e Próximos Passos**Resumo da EDA:**- Dados de qualidade: 2081 registros, sem missing values significativos- Tendência de alta com clusters de volatilidade- Correlação moderada entre ações — útil para diversificação- Sazonalidade mensal observável — features de calendário recomendadas- Caudas pesadas — modelo deve ser robusto a outliers**Próximos passos:**1. Feature engineering com indicadores técnicos (RSI, MACD, Bollinger)2. Otimização de hiperparâmetros do LSTM com Optuna3. Treinamento do baseline e registro no MLflow4. Construção do agente com tools baseadas nesses dados